In [ ]:
# -*- coding: utf-8 -*-
"""
mesh_data_processor.py - MeSH vocabulary extraction and stop-word generation (pipeline Step 1).

Parses the NLM MeSH descriptor XML (e.g. desc2025.xml) in a single,
memory-efficient streaming pass and produces the support files the rest of the
pipeline depends on.

Specifically it:
  - extracts every descriptor's name, unique identifier, and tree numbers;
  - derives a stop-word list by excluding terms that fall outside the relevant
    MeSH categories (Anatomy, Diseases, Chemicals/Drugs, Phenomena) and writes
    it as an importable Python module;
  - exports a CSV of unique terms, each tagged with a stop-word flag.
"""

import os
import traceback
import xml.etree.ElementTree as ET
from collections import defaultdict
import pandas as pd
from pathlib import Path

# <<< Constants >>>
CATEGORIES_TO_KEEP = {'A', 'C', 'D', 'G'}
CHECK_TAGS_TO_STOP = {'Male', 'Female'}

TOP_LEVEL_CATEGORY_NAMES = {
    'A': "Anatomy", 'B': "Organisms", 'C': "Diseases", 'D': "Chemicals and Drugs",
    'E': "Analytical, Diagnostic and Therapeutic Techniques, and Equipment",
    'F': "Psychiatry and Psychology", 'G': "Phenomena and Processes",
    'H': "Disciplines and Occupations",
    'I': "Anthropology, Education, Sociology, and Social Phenomena",
    'J': "Technology, Industry, Agriculture", 'K': "Humanities",
    'L': "Information Science", 'M': "Named Groups", 'N': "Health Care",
    'V': "Publication Characteristics", 'Z': "Geographicals"
}

def _get_missing_nlm_file_message(expected_path: str) -> str:
    """Generates a detailed error message for missing NLM XML data."""
    return (
        f"Raw MeSH XML File Not Found at: {expected_path}\n"
        "To resolve this:\n"
        "1. Go to the NLM Data Distribution page: https://nlmpubs.nlm.nih.gov/projects/mesh/2025/xmlmesh/\n"
        "2. Download the 'desc2025.xml' file.\n"
        "3. Place it in your raw data directory and ensure the configuration matches."
    )

def write_stopwords_to_python(output_path: str, grouped_stopwords: dict, missing_tns: list) -> bool:
    """Writes the categorized stop words to a properly formatted Python module."""
    cats_display_list = []

    for cat in sorted(list(CATEGORIES_TO_KEEP)):
        name = TOP_LEVEL_CATEGORY_NAMES.get(cat, "Unknown")
        cats_display_list.append(f"'{cat} - {name}'")

    cats_formatted_str = "[" + ", ".join(cats_display_list) + "]"

    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write('"""\nCombined MeSH Stop Word List\n')
            f.write('Generated by extracting Tree Numbers from MeSH XML file.\n"""\n\n')
            f.write("MESH_STOP_WORDS = [\n")
            f.write("    # This list contains MeSH terms that generally do NOT fall into the primary\n")
            f.write(f"    # MeSH categories: {cats_formatted_str}.\n")
            f.write("    # It includes:\n")
            f.write("    #   1. Terms explicitly categorized outside of the 'kept' categories.\n")
            f.write("    #   2. Terms for which categories could not be determined.\n\n")
            f.write("    # --- Terms Categorized Outside A, C, D, G ---\n")

            sorted_cats = sorted(grouped_stopwords.keys())

            for cat in sorted_cats:
                terms = sorted(list(grouped_stopwords[cat]))
                cat_desc = TOP_LEVEL_CATEGORY_NAMES.get(cat, "Unknown Category")

                f.write("\n")
                f.write("    #---------------------------------------------------------------------------\n")
                f.write(f"    # MeSH Category (source of these stop words): {cat} - {cat_desc}\n")
                f.write("    #---------------------------------------------------------------------------\n")

                for term in terms:
                    clean_term = term.replace('\\', '\\\\').replace('"', '\\"')
                    f.write(f'    "{clean_term}",\n')

            if missing_tns:
                f.write("\n")
                f.write("    #---------------------------------------------------------------------------\n")
                f.write("    # Terms With Undetermined Categories (No valid Tree Numbers found)\n")
                f.write("    # These are included as stop words as their category could not be confirmed to be A,C,D,G.\n")
                f.write("    #---------------------------------------------------------------------------\n")

                for term in missing_tns:
                    clean_term = term.replace('\\', '\\\\').replace('"', '\\"')
                    f.write(f'    "{clean_term}",\n')

            f.write("]\n")

        print(f"    [+] Successfully generated stop words file: {output_path}")
        return True

    except Exception as e:
        raise RuntimeError(f"Error writing Python stop word file: {e}")

def extract_all_mesh_data_from_xml(xml_file_path: str, output_csv_path: str, output_py_path: str) -> bool:
    """
    Reads the MeSH XML file in a single pass.
    Extracts Headings, UIs, and Tree Numbers, and generates both the CSV and PY files.
    """
    print(f"\n<<< Starting Unified XML Extraction >>>")
    print(f"Reading from: {xml_file_path}")

    if not os.path.exists(xml_file_path):
        raise FileNotFoundError(_get_missing_nlm_file_message(xml_file_path))

    mesh_terms = []
    mesh_term_to_categories = defaultdict(set)
    all_mh_found = set()

    try:
        # Keep a handle on the root so we can drop already-parsed siblings; without
        # this, iterparse retains every emptied record and memory grows unbounded.
        context = ET.iterparse(xml_file_path, events=('start', 'end'))
        _, root = next(context)
        count = 0

        for event, elem in context:
            if event != 'end' or elem.tag != 'DescriptorRecord':
                continue

            count += 1
            if count % 2000 == 0:
                print(f"  Parsed {count:,} descriptor records...", end='\r')

            ui = elem.findtext('.//DescriptorUI')
            name = elem.findtext('.//DescriptorName/String')

            if ui and name:
                mesh_terms.append([name, ui])
                all_mh_found.add(name)

                tree_list = elem.find('.//TreeNumberList')
                if tree_list is not None:
                    for tree_node in tree_list.findall('TreeNumber'):
                        tree_num = tree_node.text
                        if tree_num and tree_num[0].isalpha():
                            top_cat = tree_num[0].upper()
                            mesh_term_to_categories[name].add(top_cat)

            elem.clear()
            root.clear()

        print(f"  Parsed {count:,} total records. Analyzing hierarchies...\n")

        # --- 1. PROCESS STOP WORDS ---
        stop_words_grouped = defaultdict(set)
        terms_missing_tns = sorted(list(all_mh_found - set(mesh_term_to_categories.keys())))
        final_orphans_set = set(terms_missing_tns)

        for mh, cats in mesh_term_to_categories.items():
            if mh in CHECK_TAGS_TO_STOP:
                final_orphans_set.add(mh)
                continue

            if not any(c in CATEGORIES_TO_KEEP for c in cats):
                excluded_cats = sorted(list(cats))
                primary_cat = excluded_cats[0] if excluded_cats else 'Unknown'
                stop_words_grouped[primary_cat].add(mh)

        for tag in CHECK_TAGS_TO_STOP:
            if tag in all_mh_found:
                 final_orphans_set.add(tag)

        all_stop_words = final_orphans_set.copy()
        for s in stop_words_grouped.values():
            all_stop_words.update(s)

        write_stopwords_to_python(output_py_path, stop_words_grouped, sorted(list(final_orphans_set)))

        # --- 2. PROCESS DATAFRAME & TAG CSV ---
        print(f"\n<<< Building DataFrame & Tagging Stop Words >>>")
        df = pd.DataFrame(mesh_terms, columns=["DescriptorName", "DescriptorUI"])
        initial_len = len(df)

        df.drop_duplicates(subset=['DescriptorName'], keep='first', inplace=True)

        stop_words_lower = {s.lower() for s in all_stop_words}
        df['MeSH_stop_term'] = df['DescriptorName'].astype(str).str.lower().isin(stop_words_lower)
        stop_count = df['MeSH_stop_term'].sum()

        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
        df.to_csv(output_csv_path, index=False)

        print(f"    [+] Successfully extracted {len(df):,} unique terms (from {initial_len:,} raw)")
        print(f"    [+] Tagged {stop_count:,} terms as stop words.")
        print(f"    [+] Saved CSV to '{output_csv_path}'")

        return True

    except MemoryError:
        raise RuntimeError("Memory Limit Exceeded during XML processing.")
    except Exception as e:
        traceback.print_exc()
        raise RuntimeError(f"Unexpected error during XML extraction: {e}")


def process_raw_mesh_data(xml_file: str, output_csv: str, output_py: str, force_update: bool = False):
    """
    Orchestrates the unified XML extraction and stop word generation process.
    """
    print("--- MeSH Data Processor ---")
    print(f"XML Input:    {xml_file}")
    print(f"CSV Output:   {output_csv}")
    print(f"PY Output:    {output_py}")
    print(f"Force Update: {force_update}")

    print("\n<<< Starting Processing >>>")

    # Because XML generates both files at the exact same time, we check if EITHER is missing
    needs_processing = not os.path.exists(output_csv) or not os.path.exists(output_py) or force_update

    if needs_processing:
        extract_all_mesh_data_from_xml(xml_file, output_csv, output_py)
    else:
        print("    Skipping XML extraction (Both CSV and PY exist, and Force Update is OFF).")

    print("\n<<< Data Processing Complete >>>")